In [1]:
import numpy as np 
import pandas as pd

In [2]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv('u.data', sep='\t', names = column_names)

In [3]:
df.head()

,user_id,item_id,rating,timestamp
0,0,50,5,881250949
1,0,172,5,881250949
2,0,133,1,881250949
3,196,242,3,881250949
4,186,302,3,891717742


In [4]:
movie_titles = pd.read_csv('Movie_Id_Titles')
movie_titles.head()

,item_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [5]:
df = pd.merge(df,movie_titles,on='item_id')
df.head()

,user_id,item_id,rating,timestamp,title
0,0,50,5,881250949,Star Wars (1977)
1,0,172,5,881250949,"Empire Strikes Back, The (1980)"
2,0,133,1,881250949,Gone with the Wind (1939)
3,196,242,3,881250949,Kolya (1996)
4,186,302,3,891717742,L.A. Confidential (1997)


In [6]:
#Checking the number of users and movies in the dataset
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()

print(f'number of user: {n_users}')
print(f'number of movies: {n_items}')

number of user: 944
number of movies: 1682


In [7]:
#Splitting the data for later evaluation
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(df, test_size=0.25)

In [8]:
#Creating a user-item matrix
train_data_matrix = np.zeros((n_users, n_items))
for line in train_data.itertuples():
    train_data_matrix[line[1]-1,line[2]-1] = line[3]

test_data_matrix = np.zeros((n_users, n_items))
for line in test_data.itertuples():
    test_data_matrix[line[1]-1,line[2]-1] = line[3] 


In [9]:
train_data.head() #smjhnay k liy 

,user_id,item_id,rating,timestamp,title
83310,1,3,4,878542960,Four Rooms (1995)
40846,645,175,5,892054537,Brazil (1985)
29411,222,655,4,878182210,Stand by Me (1986)
1246,43,40,3,883956468,"To Wong Foo, Thanks for Everything! Julie Newm..."
43859,327,152,3,887819194,Sleeper (1973)


In [10]:
# our user-item matrices, Row: User id, Column: Movie id, Value: the rating user gave
test_data_matrix

array([[0., 0., 0., ..., 0., 0., 0.],
       [4., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(944, 1682))

In [11]:
train_data_matrix

array([[5., 3., 4., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 5., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(944, 1682))

In [12]:
# Cosine similarity(Pairwise function from sklearn), to find the similarity between movies accoring with users. 
# key idea: to find out which movies are similar in user perspective

from sklearn.metrics.pairwise import pairwise_distances
user_similarity =  pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric = 'cosine')

In [13]:
# Doing user based prediction and doing item based prediction

def predict(ratings, similarity, type='user'):
    if type == 'user':
        mean_user_rating = ratings.mean(axis=1)
        rating_diff = (ratings - mean_user_rating[:, np.newaxis])
        pred = mean_user_rating[:, np.newaxis] + similarity.dot(rating_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
    elif type == 'item':
        pred = ratings.dot(similarity)/ np.array([np.abs(similarity).sum(axis=1)])
    return pred
        

In [14]:
#Making Predictions

item_prediction = predict(train_data_matrix, item_similarity, type='item')
user_prediction = predict(train_data_matrix, user_similarity, type='user')

In [15]:
# User 10 (index 9) ki Movie 50 (index 49) ki prediction
specific_rating = user_prediction[9, 49]
print(specific_rating) 
# Output shayad aayega: 4.2

2.0896981559383168


In [16]:
#Doing evaluation use RMSE 
from sklearn.metrics import mean_squared_error
from math import sqrt
def rmse(prediction, ground_truth):
    prediction = prediction[ground_truth.nonzero()].flatten()
    ground_truth = ground_truth[ground_truth.nonzero()].flatten()
    return sqrt(mean_squared_error(prediction, ground_truth))
    

In [17]:
print("User-based CF RMSE: "+ str(rmse(user_prediction, test_data_matrix)))
print("Item-based CF RMSE: "+ str(rmse(item_prediction, test_data_matrix)))

User-based CF RMSE: 3.1301538927610357
Item-based CF RMSE: 3.4569768656169924


In [18]:
#Model Based Collaborative system

#Calculating Sparsity
sparsity = round(1.0 - len(df)/ float(n_users*n_items), 3)
print('The sparsity level of MovieLens dataset is: '+ str(sparsity*100)+'%')

The sparsity level of MovieLens dataset is: 93.7%


In [ ]:
# SVD Predictions
import scipy.sparse as sp
from scipy.sparse.linalg import svds

#Getting svf from train matrix. k will be random(k = number of latent features we want)
u, s, vt = svds(train_data_matrix, k=20)
s_diag_matrix = np.diag(s)
X_pred